# Phase 5 Notebook: Multi-Gameweek Optimisation

## What was done
- Implemented PuLP MILP squad optimizer with legal FPL squad constraints.
- Added discounted multi-gameweek expected return objective with risk adjustment.
- Added transfer penalty handling with free-transfer offsets and chip bonus support.

## Why it was done
- To optimize decision quality beyond one-week point maximization.
- To model realistic FPL decision constraints and transfer economics.

In [ ]:
import json
from pathlib import Path

from fpl_ai_agent.contracts import OptimiserInputContract
from fpl_ai_agent.optimisation.problem import OptimisationSettings, TransferContext, optimize_squad

rows = json.loads(Path('../tests/fixtures/optimiser_candidates.json').read_text(encoding='utf-8'))
candidates = [OptimiserInputContract(**row) for row in rows]
settings = OptimisationSettings(budget=100.0, horizon_weeks=3)
context = TransferContext(current_squad_ids=set(), free_transfers=1, bench_boost_available=True, bench_boost_bonus=1.0)
result = optimize_squad(candidates, settings=settings, transfer_context=context)
result

## Data quality checks
- Position quotas are exactly met (2 GK, 5 DEF, 5 MID, 3 FWD).
- Team cap (max 3 from one club) is respected.
- Budget cap is respected.
- Objective decomposes into expected value, risk, and transfer penalties.

In [ ]:
selected = [c for c in candidates if c.player_id in set(result.selected_player_ids)]
dq = {
    'selected_count': len(selected),
    'gk_count': sum(c.position == 'GK' for c in selected),
    'def_count': sum(c.position == 'DEF' for c in selected),
    'mid_count': sum(c.position == 'MID' for c in selected),
    'fwd_count': sum(c.position == 'FWD' for c in selected),
    'total_cost': round(sum(c.cost for c in selected), 4),
    'max_team_count': max(sum(c.team == t for c in selected) for t in {c.team for c in selected}),
    'chip_used': result.chip_used,
}
dq

## Findings and anomalies
- The optimizer can bias toward high-uncertainty high-upside players when risk aversion is low.
- Transfer penalties strongly alter decisions when free transfers are exhausted.

## How anomalies were handled
- Included an explicit risk term with configurable aversion.
- Included paid-transfer penalties and chip bonus controls in the objective.
- Enforced hard legal constraints to prevent invalid squad recommendations.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

selected_df = pd.DataFrame(
    [
        {
            "player_id": c.player_id,
            "position": c.position,
            "team": c.team,
            "cost": c.cost,
            "expected_w1": c.expected_points_horizon[0] if c.expected_points_horizon else 0.0,
            "uncertainty_w1": c.uncertainty_horizon[0] if c.uncertainty_horizon else 0.0,
        }
        for c in selected
    ]
)

position_perf = selected_df.groupby("position", as_index=False)["expected_w1"].sum()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(position_perf["position"], position_perf["expected_w1"], color="#4e79a7")
axes[0].set_title("Selected Squad Expected Points by Position (W1)")
axes[0].set_xlabel("Position")
axes[0].set_ylabel("Expected Points")

scatter = axes[1].scatter(
    selected_df["cost"],
    selected_df["expected_w1"],
    c=selected_df["uncertainty_w1"],
    cmap="magma",
    s=70,
    alpha=0.85,
)
axes[1].set_title("Cost vs Expected Points (Color: Uncertainty)")
axes[1].set_xlabel("Cost")
axes[1].set_ylabel("Expected Points (W1)")
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label("uncertainty_w1")

plt.tight_layout()
plt.show()

objective_breakdown = {
    "expected_points_value": round(result.expected_points_value, 3),
    "risk_value": round(result.risk_value, 3),
    "objective_value": round(result.objective_value, 3),
    "paid_transfers": round(result.paid_transfers, 3),
}
objective_breakdown